## Setup

In [5]:
import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
from utils import chat_interface

In [6]:
load_dotenv()

True

## Run

In [8]:
import os                                                                                                                            
os.remove("data/core/memory.db")                                                                                                     
print("Cleared stale checkpoints") 

Cleared stale checkpoints


In [9]:
!rm -f data/core/memory.db

In [1]:
from agentic.workflow import create_orchestrator

orchestrator = create_orchestrator()

In [ ]:
await chat_interface(orchestrator, ticket_id="ticket-001", customer_id="f556c0")

In [ ]:
history = [state async for state in orchestrator.aget_state_history(
    config={"configurable": {"thread_id": "ticket-001"}}
)]
history[0].values["messages"]

## Test Cases

Each test sends a single message through the full pipeline:
classification → routing → retrieval → tool usage → resolution or escalation.

We use `ainvoke` directly (not `chat_interface`) so results are captured programmatically.

In [3]:
from langchain_core.messages import HumanMessage
import re

async def run_test(test_name, message, ticket_id, customer_id="f556c0"):
    """Run a single test case and display the result."""
    print(f"\n{'='*60}")
    print(f"TEST: {test_name}")
    print(f"Input: {message}")
    print(f"{'='*60}")
    
    result = await orchestrator.ainvoke(
        {
            "messages": [HumanMessage(content=message)],
            "ticket_id": ticket_id,
            "customer_id": customer_id,
        },
        config={"configurable": {"thread_id": ticket_id}}
    )
    
    response = result["messages"][-1].content
    print(f"\nResponse:\n{response}")
    
    # Extract confidence level from the response
    confidence_match = re.search(r"\[Confidence:\s*(HIGH|MEDIUM|LOW)\]", response)
    if confidence_match:
        print(f"\n--- Confidence: {confidence_match.group(1)} ---")
    else:
        print(f"\n--- Outcome: Escalated to human specialist ---")
    
    return result

### Test 1: Resolution — Billing (refund inquiry)
Routes to billing agent, looks up subscription/payment history, finds KB article, resolves with high confidence.

In [3]:
result_billing = await run_test(
    "Resolution — Billing refund inquiry",
    "I was charged twice for my subscription this month. My user ID is 88382b. Can I get a refund?",
    ticket_id="test-billing-002",
    customer_id="88382b",
)


TEST: Resolution — Billing refund inquiry
Input: I was charged twice for my subscription this month. My user ID is 88382b. Can I get a refund?

Response:
It appears that there are no duplicate charges for your subscription this month. Your current subscription is active under the premium tier, and billing information shows that you are only being charged once as of the last update.

If you're still seeing double charges on your bank statement, I recommend contacting your bank for more specific details.

[Confidence: HIGH]

--- Indicators ---
Confidence score present: True
Escalation signaled: False


### Test 2: Resolution — General (FAQ)
Routes to general agent, finds KB article about gift cards, resolves with high confidence.

In [4]:
result_general = await run_test(
    "Resolution — General FAQ",
    "How do I use a CultPass gift card? I received one as a birthday present.",
    ticket_id="test-general-001",
)


TEST: Resolution — General FAQ
Input: How do I use a CultPass gift card? I received one as a birthday present.

Response:
You can redeem your CultPass gift card in the app by following these steps:

1. Go to **'My Account'** and select **'Redeem Gift Card'**.
2. Enter the **16-digit code** from your gift card.
3. The gift card credit will be applied to your next subscription payment.

Please note that gift card credit never expires, but it cannot be converted to cash. If you encounter any issues with the code, ensure it hasn't already been redeemed and check for any typos. If the code still doesn't work, you may need to contact support with the gift card details.

[Confidence: HIGH]

--- Indicators ---
Confidence score present: True
Escalation signaled: False


### Test 3: Resolution — Technical (app crash)
Routes to technical agent, looks up ticket history, finds KB troubleshooting article, resolves.

In [8]:
result_technical = await run_test(
    "Resolution — Technical app crash",
    "The CultPass app keeps crashing when I try to open my reservations. I'm on Android 14.",
    ticket_id="test-technical-001",
)


TEST: Resolution — Technical app crash
Input: The CultPass app keeps crashing when I try to open my reservations. I'm on Android 14.

Response:
It appears that the CultPass app crashing issue has been documented in the knowledge base. Here are the steps to troubleshoot the app crashing on your Android device:

1. **Update the App**: Ensure you are using the latest version of the CultPass app. Check for updates in the Google Play Store and install any available updates.

2. **Clear App Cache**:
   - Go to your device's **Settings**.
   - Tap on **Apps**.
   - Find and select **CultPass**.
   - Tap on **Storage**.
   - Select **Clear Cache**. 

3. **Restart Your Device**: After clearing the cache, restart your phone.

4. **Check for Known Outages**: Occasionally, app functions may be affected by outages, so check the CultPass status page for any ongoing issues.

5. **Escalate if Needed**: If the problem persists after these steps, please provide your device model and any additional deta

### Test 4: Escalation — No relevant KB article (low confidence)
The agent cannot find a relevant article, reports low confidence, and signals escalation.

In [4]:
result_escalation_low = await run_test(
    "Escalation — Low confidence (no KB match)",
    "I need to transfer my CultPass subscription to another country. I'm relocating to Japan next month.",
    ticket_id="test-escalation-001",
)


TEST: Escalation — Low confidence (no KB match)
Input: I need to transfer my CultPass subscription to another country. I'm relocating to Japan next month.

Response:
It seems there is no active CultPass subscription associated with your account, which means you currently may not be able to transfer a subscription to Japan. If you are planning to establish a new subscription there, you can either create a new account or simply let me know if any changes occur regarding this.

Unfortunately, I couldn't find specific information on transferring a subscription or maintaining benefits while moving to a different country in the knowledge base.

[Confidence: LOW]

--- Confidence: LOW ---


### Test 5: Escalation — Safety/discrimination emergency
High urgency, routes directly to escalation agent which creates an escalation ticket.

In [5]:
result_escalation_safety = await run_test(
    "Escalation — Safety/discrimination",
    "I was denied entry to an event because of my disability. The venue staff were rude and refused to provide wheelchair access despite my CultPass reservation. This is unacceptable.",
    ticket_id="test-escalation-002",
)


TEST: Escalation — Safety/discrimination
Input: I was denied entry to an event because of my disability. The venue staff were rude and refused to provide wheelchair access despite my CultPass reservation. This is unacceptable.

Response:
Dear Bob,

I sincerely apologize for the unacceptable experience you had regarding access to the event. I understand how frustrating it must be to face such discrimination, especially when you were relying on your CultPass reservation.

I have escalated your issue to a human support specialist, who will follow up with you shortly regarding your situation. You can expect to hear back from them within 24 hours.

Thank you for your patience, and please know that your concerns are being taken seriously.

Best regards,  
CultPass Support Team

--- Outcome: Escalated to human specialist ---


### Verify: Structured Logging
Check that tool usage was logged to JSON files.

In [6]:
import json
from pathlib import Path

logs_dir = Path("logs")
log_files = sorted(logs_dir.glob("session_*.json"))
print(f"Log files found: {len(log_files)}")
for f in log_files:
    entries = json.loads(f.read_text())
    print(f"\n{f.name}: {len(entries)} entries")
    for entry in entries[:3]:  # show first 3 entries
        print(f"  [{entry['timestamp'][:19]}] {entry['tool_name']} → {entry['output'][:80]}...")

Log files found: 22

session_20260313_012222.json: 96 entries
  [2026-03-13T01:22:23] route_to_billing → invoking billing agent...
  [2026-03-13T01:22:25] billing_agent_response → I can help with that. To start, I will look up your payment history to identify ...
  [2026-03-13T01:22:26] route_to_billing → invoking billing agent...

session_20260313_012924.json: 4 entries
  [2026-03-13T01:29:27] route_to_billing → invoking billing agent...
  [2026-03-13T01:29:34] billing_agent_response → It appears that I couldn't find any billing history or subscription information ...
  [2026-03-13T01:29:37] route_to_escalation → invoking escalation agent...

session_20260313_202617.json: 3 entries
  [2026-03-13T20:28:34] route_to_general → invoking general agent...
  [2026-03-13T20:28:35] lookup_customer_history → [{'type': 'text', 'text': '{\n  "external_user_id": "user-1",\n  "tickets": [],\...
  [2026-03-13T20:28:40] general_agent_response → You can redeem your CultPass gift card in the app by fol

### Verify: Short-term Memory (State Persistence)
Inspect the conversation state saved by SqliteSaver.

In [ ]:
# Inspect persisted state for the escalation test
history = [state async for state in orchestrator.aget_state_history(
    config={"configurable": {"thread_id": "test-escalation-002"}}
)]
print(f"States saved: {len(history)}")
print(f"\nFinal message:\n{history[0].values['messages'][-1].content[:300]}...")

States saved: 6

Final message:
Dear Bob,

I sincerely apologize for the unacceptable experience you had regarding access to the event. I understand how frustrating it must be to face such discrimination, especially when you were relying on your CultPass reservation.

I have escalated your issue to a human support specialist, who ...
